<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Python profesional para analítica avanzada</p>
  </div>

</div>

# **Logging, excepciones y refactorización**

En este laboratorio aprenderemos a construir pipelines **trazables y diagnosticables**.

Metodología:

> **provocar → observar → diagnosticar → corregir → validar**

Trabajaremos `logging`, niveles, logs a fichero, excepciones específicas, `try/except/else/finally`, excepciones de dominio, relanzado y refactorización.

## Objetivos

Al finalizar deberías poder:

1. Explicar cuándo `print()` deja de ser suficiente.
2. Configurar un logger.
3. Elegir niveles adecuados.
4. Persistir logs a fichero.
5. Capturar excepciones específicas.
6. Evitar `except:` indiscriminado.
7. Utilizar `else` y `finally`.
8. Crear excepciones propias.
9. Relanzar errores conservando contexto.
10. Construir un pipeline completamente trazable.

## 1. Preparación

In [ ]:
from pathlib import Path
import logging
import time
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
rng = np.random.default_rng(SEED)

print("Entorno preparado.")

## 2. Empezar con `print()`

In [ ]:
def proceso_con_print(datos):
    print("Inicio")
    print("Filas:", len(datos))
    salida = datos.copy()
    salida["ratio"] = salida["gasto"] / salida["ingresos"]
    print("Transformación completada")
    print("Fin")
    return salida

demo = pd.DataFrame({
    "ingresos": [30000, 45000, 52000],
    "gasto": [12000, 18000, 22000]
})

proceso_con_print(demo)

### Limitaciones

`print()` no ofrece de forma natural:

- severidad;
- timestamp;
- persistencia;
- filtrado;
- nombre del componente;
- integración con monitorización.

## 3. Configurar logging

In [ ]:
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    force=True
)

logger = logging.getLogger("data_carpentry")

logger.debug("Mensaje DEBUG")
logger.info("Mensaje INFO")
logger.warning("Mensaje WARNING")
logger.error("Mensaje ERROR")
logger.critical("Mensaje CRITICAL")

### Niveles

- `DEBUG`: detalle técnico.
- `INFO`: funcionamiento normal.
- `WARNING`: situación inesperada pero recuperable.
- `ERROR`: operación fallida.
- `CRITICAL`: fallo grave.

## 4. Sustituir `print()` por logging

In [ ]:
def proceso_con_logging(datos):
    logger.info("Inicio del proceso")
    logger.debug("Filas recibidas: %d", len(datos))

    salida = datos.copy()
    salida["ratio"] = salida["gasto"] / salida["ingresos"]

    logger.info("Transformación completada")
    logger.info("Fin del proceso")
    return salida

proceso_con_logging(demo)

## 5. Filtrar por nivel

In [ ]:
logger.setLevel(logging.WARNING)

logger.debug("No visible")
logger.info("No visible")
logger.warning("Visible")
logger.error("Visible")

logger.setLevel(logging.DEBUG)

## 6. Registrar variables

In [ ]:
n_filas = 1250
n_nulos = 17

logger.info(
    "Dataset cargado: filas=%d, nulos=%d",
    n_filas,
    n_nulos
)

## 7. Inconsistencias: primero observar

In [ ]:
datos_calidad = pd.DataFrame({
    "cliente_id": [1, 2, 3, 4, 5],
    "ingresos": [30000, np.nan, 45000, -500, 52000],
    "gasto": [12000, 15000, -1000, 18000, 22000]
})

datos_calidad

### 7.1 Diagnóstico

In [ ]:
diagnostico = {
    "nulos_ingresos": datos_calidad["ingresos"].isna().sum(),
    "ingresos_negativos": (datos_calidad["ingresos"] < 0).sum(),
    "gastos_negativos": (datos_calidad["gasto"] < 0).sum()
}

diagnostico

In [ ]:
logger.info("Nulos en ingresos: %d", diagnostico["nulos_ingresos"])

if diagnostico["ingresos_negativos"] > 0:
    logger.warning(
        "Ingresos negativos detectados: %d",
        diagnostico["ingresos_negativos"]
    )

if diagnostico["gastos_negativos"] > 0:
    logger.warning(
        "Gastos negativos detectados: %d",
        diagnostico["gastos_negativos"]
    )

### 7.2 Visualizar antes de corregir

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(datos_calidad["ingresos"], datos_calidad["gasto"])
ax.axvline(0, linestyle="--")
ax.axhline(0, linestyle="--")
ax.set(title="Datos ANTES de limpiar", xlabel="Ingresos", ylabel="Gasto")
plt.show()

**Interpretación.** Los valores negativos quedan en regiones conceptualmente inválidas. El notebook los hace visibles antes de modificar nada.

## 8. Limpiar dejando trazabilidad

In [ ]:
def limpiar_clientes(datos):
    logger.info("Inicio de limpieza")
    salida = datos.copy()

    n = salida["ingresos"].isna().sum()
    if n:
        logger.warning("Imputando %d ingresos nulos", n)
        salida["ingresos"] = salida["ingresos"].fillna(
            salida["ingresos"].median()
        )

    n = (salida["ingresos"] < 0).sum()
    if n:
        logger.warning("Corrigiendo %d ingresos negativos", n)
        salida.loc[salida["ingresos"] < 0, "ingresos"] = np.nan
        salida["ingresos"] = salida["ingresos"].fillna(
            salida["ingresos"].median()
        )

    n = (salida["gasto"] < 0).sum()
    if n:
        logger.warning("Corrigiendo %d gastos negativos", n)
        salida.loc[salida["gasto"] < 0, "gasto"] = np.nan
        salida["gasto"] = salida["gasto"].fillna(
            salida["gasto"].median()
        )

    logger.info("Limpieza completada")
    return salida

datos_limpios = limpiar_clientes(datos_calidad)
datos_limpios

### 8.1 Volver a validar

In [ ]:
diagnostico_despues = {
    "nulos_ingresos": datos_limpios["ingresos"].isna().sum(),
    "ingresos_negativos": (datos_limpios["ingresos"] < 0).sum(),
    "gastos_negativos": (datos_limpios["gasto"] < 0).sum()
}

pd.DataFrame({
    "antes": diagnostico,
    "despues": diagnostico_despues
})

### 8.2 Visualización antes/después

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.scatter(
    datos_calidad["ingresos"],
    datos_calidad["gasto"],
    label="Antes",
    alpha=0.7
)

ax.scatter(
    datos_limpios["ingresos"],
    datos_limpios["gasto"],
    label="Después",
    alpha=0.7
)

ax.axvline(0, linestyle="--")
ax.axhline(0, linestyle="--")
ax.set(title="Antes vs después", xlabel="Ingresos", ylabel="Gasto")
ax.legend()
plt.show()

**Interpretación.** La validación cuantitativa y el gráfico confirman el efecto de la limpieza. El log conserva la secuencia de decisiones aplicada.

## 9. Escribir logs a fichero

In [ ]:
log_dir = Path("logs_demo")
log_dir.mkdir(exist_ok=True)

log_file = log_dir / "pipeline.log"

file_handler = logging.FileHandler(
    log_file,
    mode="w",
    encoding="utf-8"
)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(
    logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    )
)

logger.addHandler(file_handler)

logger.info("Logging a fichero activado.")
logger.warning("Warning de demostración.")

print(log_file.resolve())

### 9.1 Leer el log

In [ ]:
print(log_file.read_text(encoding="utf-8"))

### 9.2 Inspeccionar handlers

In [ ]:
for handler in logger.handlers:
    print(type(handler).__name__)

En notebooks hay que evitar añadir el mismo handler repetidamente si una celda se reejecuta.

# 10. Excepciones: identificar primero el error real

In [ ]:
def dividir(a, b):
    return a / b

try:
    dividir(10, 0)
except Exception as error:
    print(type(error).__name__, "->", error)

El error concreto es `ZeroDivisionError`. Capturar primero el tipo real ayuda a diseñar una gestión adecuada.

## 11. Captura específica

In [ ]:
def dividir_seguro(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        logger.error("División por cero: a=%s, b=%s", a, b)
        raise

try:
    dividir_seguro(10, 0)
except ZeroDivisionError:
    print("Error registrado y propagado.")

## 12. Mala práctica: `except Exception` que oculta

In [ ]:
def cargar_archivo_mal(ruta):
    try:
        return pd.read_csv(ruta)
    except Exception:
        print("Algo salió mal")
        return None

resultado = cargar_archivo_mal("archivo_que_no_existe.csv")
print("Resultado:", resultado)

### Problema

La función oculta el tipo de error y devuelve `None`. El pipeline podría continuar y fallar mucho más tarde.

## 13. Capturas específicas al leer CSV

In [ ]:
def cargar_csv_seguro(ruta: Path) -> pd.DataFrame:
    try:
        datos = pd.read_csv(ruta)

    except FileNotFoundError:
        logger.error("No existe el fichero: %s", ruta)
        raise

    except pd.errors.EmptyDataError:
        logger.error("CSV vacío: %s", ruta)
        raise

    except pd.errors.ParserError:
        logger.error("CSV mal formado: %s", ruta)
        raise

    else:
        logger.info(
            "CSV cargado correctamente: %s | filas=%d",
            ruta,
            len(datos)
        )
        return datos

### 13.1 Provocar FileNotFoundError

In [ ]:
try:
    cargar_csv_seguro(Path("no_existe.csv"))
except FileNotFoundError:
    print("FileNotFoundError gestionado correctamente.")

## 14. `else`

El bloque `else` se ejecuta únicamente cuando `try` termina sin excepciones. Ayuda a separar el código susceptible de fallar del código posterior.

## 15. `finally`

In [ ]:
def demostrar_finally():
    try:
        print("Inicio")
        raise ValueError("Error simulado")
    except ValueError as error:
        print("Capturado:", error)
    finally:
        print("FINALLY: se ejecuta siempre")

demostrar_finally()

## 16. Context manager

In [ ]:
ruta_demo = Path("context_manager_demo.txt")

with ruta_demo.open("w", encoding="utf-8") as fichero:
    fichero.write("Recurso gestionado con context manager.")

print(ruta_demo.read_text(encoding="utf-8"))

`with` garantiza el cierre del recurso y suele ser preferible a gestionar manualmente apertura/cierre con `finally`.

## 17. Encadenar excepciones

In [ ]:
def convertir_entero(valor: str) -> int:
    try:
        return int(valor)
    except ValueError as error:
        raise ValueError(
            f"No se puede convertir {valor!r} a entero."
        ) from error

try:
    convertir_entero("abc")
except ValueError as error:
    print(type(error).__name__, "->", error)

## 18. Excepción de dominio

In [ ]:
class DatosInvalidosError(Exception):
    """Datos que incumplen reglas del pipeline."""
    pass

In [ ]:
def validar_importes(datos):
    if "importe" not in datos.columns:
        raise DatosInvalidosError(
            "Falta la columna 'importe'."
        )

    if (datos["importe"] < 0).any():
        raise DatosInvalidosError(
            "Se han detectado importes negativos."
        )

### 18.1 Provocar el error

In [ ]:
demo_error = pd.DataFrame({
    "importe": [10, -5, 20]
})

try:
    validar_importes(demo_error)
except DatosInvalidosError as error:
    logger.error("Validación fallida: %s", error)
    print(type(error).__name__, "->", error)

Una excepción propia distingue fallos del dominio de fallos técnicos de Python o de librerías.

# 19. Pipeline trazable completo

In [ ]:
def validar_ventas(datos):
    logger.info("Etapa VALIDACIÓN")

    requeridas = {"venta_id", "unidades", "precio", "canal"}
    faltantes = requeridas - set(datos.columns)

    if faltantes:
        raise DatosInvalidosError(
            f"Faltan columnas: {sorted(faltantes)}"
        )

    if (datos["unidades"] <= 0).any():
        raise DatosInvalidosError(
            "Las unidades deben ser positivas."
        )

    if (datos["precio"] < 0).any():
        raise DatosInvalidosError(
            "El precio no puede ser negativo."
        )

    logger.info("Validación correcta")

In [ ]:
def preparar_ventas(datos):
    logger.info("Etapa PREPARACIÓN")

    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]

    logger.debug(
        "Importe medio: %.2f",
        salida["importe"].mean()
    )

    return salida

In [ ]:
def resumir_ventas(datos):
    logger.info("Etapa ANÁLISIS")

    resumen = (
        datos.groupby("canal")
        .agg(
            operaciones=("venta_id", "size"),
            facturacion=("importe", "sum"),
            ticket_medio=("importe", "mean")
        )
        .reset_index()
    )

    logger.info("Resumen generado: %d filas", len(resumen))
    return resumen

## 19.1 Dataset correcto

In [ ]:
n = 1000

ventas = pd.DataFrame({
    "venta_id": np.arange(1, n + 1),
    "unidades": rng.integers(1, 8, n),
    "precio": rng.uniform(5, 200, n),
    "canal": rng.choice(["web", "tienda", "partner"], n)
})

ventas.head()

## 19.2 Orquestador

In [ ]:
def ejecutar_pipeline_ventas(datos):
    logger.info("=" * 50)
    logger.info("INICIO PIPELINE")

    inicio = time.perf_counter()

    try:
        validar_ventas(datos)
        preparados = preparar_ventas(datos)
        resumen = resumir_ventas(preparados)

    except DatosInvalidosError as error:
        logger.error(
            "Pipeline detenido por datos inválidos: %s",
            error
        )
        raise

    except Exception:
        logger.exception(
            "Error inesperado en el pipeline"
        )
        raise

    else:
        logger.info("Pipeline completado correctamente")
        return resumen

    finally:
        duracion = time.perf_counter() - inicio
        logger.info("Duración: %.4f s", duracion)
        logger.info("FIN PIPELINE")

## 19.3 Ejecutar correctamente

In [ ]:
resumen_ventas = ejecutar_pipeline_ventas(ventas)
resumen_ventas

### 19.4 Visualizar

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    resumen_ventas["canal"],
    resumen_ventas["facturacion"]
)

ax.set(
    title="Facturación por canal",
    xlabel="Canal",
    ylabel="Facturación"
)

plt.show()

### Interpretación

El pipeline deja trazabilidad de inicio, validación, transformación, análisis, duración y fin.

# 20. Provocar un fallo real

In [ ]:
ventas_invalidas = ventas.copy()
ventas_invalidas.loc[10, "precio"] = -50

ventas_invalidas.loc[
    [10],
    ["venta_id", "precio"]
]

## 20.1 Ejecutar antes de corregir

In [ ]:
try:
    ejecutar_pipeline_ventas(ventas_invalidas)
except DatosInvalidosError:
    print("El pipeline se detuvo como estaba previsto.")

La validación detiene el pipeline antes de generar resultados incorrectos.

## 20.2 Corregir

In [ ]:
ventas_corregidas = ventas_invalidas.copy()

mediana_precio = ventas_corregidas.loc[
    ventas_corregidas["precio"] >= 0,
    "precio"
].median()

ventas_corregidas.loc[
    ventas_corregidas["precio"] < 0,
    "precio"
] = mediana_precio

ventas_corregidas.loc[[10], ["venta_id", "precio"]]

## 20.3 Volver a ejecutar

In [ ]:
resumen_corregido = ejecutar_pipeline_ventas(
    ventas_corregidas
)

resumen_corregido

### Secuencia metodológica

1. inconsistencia;
2. observación;
3. fallo;
4. registro;
5. corrección;
6. validación;
7. ejecución exitosa.

# 21. Leer el log completo

In [ ]:
contenido_log = log_file.read_text(encoding="utf-8")
print(contenido_log[-6000:])

## 21.1 Convertir logs en una métrica

In [ ]:
lineas_log = contenido_log.splitlines()

niveles = {
    "DEBUG": 0,
    "INFO": 0,
    "WARNING": 0,
    "ERROR": 0,
    "CRITICAL": 0
}

for linea in lineas_log:
    for nivel in niveles:
        if f"| {nivel} |" in linea:
            niveles[nivel] += 1

niveles

## 21.2 Visualizar niveles

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

ax.bar(
    list(niveles.keys()),
    list(niveles.values())
)

ax.set(
    title="Mensajes por nivel",
    xlabel="Nivel",
    ylabel="Número de mensajes"
)

plt.show()

### Interpretación

Los logs también pueden convertirse en métricas. En producción, muchos `WARNING` o `ERROR` pueden revelar problemas sistemáticos.

# 22. Script problemático antes de refactorizar

In [ ]:
script_problematico = '''
import pandas as pd

print("leyendo")
try:
    df = pd.read_csv("ventas.csv")
except:
    df = None

print("limpiando")
df = df.dropna()
df["importe"] = df["unidades"] * df["precio"]

try:
    resumen = df.groupby("canal")["importe"].mean()
except:
    print("error")

print("fin")
'''

print(script_problematico)

### Diagnóstico

- `except:` indiscriminado;
- `df = None`;
- el flujo puede continuar tras un error;
- ausencia de validación;
- ausencia de funciones;
- ausencia de niveles;
- sin duración;
- sin contexto del error.

# 23. Arquitectura refactorizada

In [ ]:
print('''
pipeline/
├── io.py
├── validation.py
├── features.py
├── analysis.py
└── runner.py
''')

## 23.1 Crear estructura real

In [ ]:
raiz = Path("proyecto_logging_demo")
src = raiz / "src" / "pipeline"

src.mkdir(parents=True, exist_ok=True)

(src / "__init__.py").write_text(
    "",
    encoding="utf-8"
)

print(src.resolve())

## 23.2 Crear módulos automáticamente

In [ ]:
modulos = {
    "validation.py": """
class DatosInvalidosError(Exception):
    pass

def validar(datos):
    requeridas = {"unidades", "precio", "canal"}
    faltantes = requeridas - set(datos.columns)

    if faltantes:
        raise DatosInvalidosError(
            f"Faltan columnas: {sorted(faltantes)}"
        )
""",
    "features.py": """
def crear_importe(datos):
    salida = datos.copy()
    salida["importe"] = salida["unidades"] * salida["precio"]
    return salida
""",
    "analysis.py": """
def resumen_por_canal(datos):
    return (
        datos.groupby("canal")
        .agg(
            operaciones=("importe", "size"),
            importe_medio=("importe", "mean")
        )
        .reset_index()
    )
"""
}

for nombre, contenido in modulos.items():
    (src / nombre).write_text(
        textwrap.dedent(contenido),
        encoding="utf-8"
    )

for archivo in sorted(src.glob("*.py")):
    print("-", archivo.name)

## 23.3 Inspeccionar los módulos

In [ ]:
for archivo in sorted(src.glob("*.py")):
    print("\n---", archivo.name, "---")
    print(archivo.read_text(encoding="utf-8"))

### Interpretación

La refactorización no consiste solo en sustituir `print()` por `logger`. También separamos responsabilidades y hacemos explícitos los lugares donde pueden aparecer errores.

# 24. Checklist

- [ ] ¿Se registra inicio y fin?
- [ ] ¿Los niveles son adecuados?
- [ ] ¿Se evitan `print()` como trazabilidad?
- [ ] ¿Los warnings son recuperables?
- [ ] ¿Los errores importantes quedan registrados?
- [ ] ¿Se capturan excepciones específicas?
- [ ] ¿Se evita `except:`?
- [ ] ¿Se evita silenciar errores?
- [ ] ¿Se conserva el contexto?
- [ ] ¿Se usan excepciones de dominio cuando aportan valor?
- [ ] ¿Se gestionan recursos correctamente?
- [ ] ¿Se registra duración?
- [ ] ¿Los logs pueden persistirse?

# 25. Resumen

Un pipeline profesional debe permitir responder:

1. ¿Qué estaba haciendo?
2. ¿Qué salió mal?
3. ¿Dónde y por qué ocurrió?

`logging` aporta trazabilidad. Las excepciones definen cómo responde el programa cuando una operación no puede continuar normalmente.

# 26. Ejercicios finales

## Ejercicio 1
Sustituye cinco `print()` por niveles de logging adecuados.

## Ejercicio 2
Genera mensajes `DEBUG`, `INFO`, `WARNING` y `ERROR`, justificando cada nivel.

## Ejercicio 3
Configura un `FileHandler` y guarda una ejecución completa.

## Ejercicio 4
Introduce nulos, negativos y categorías inválidas. Regístralos antes de corregir.

## Ejercicio 5
Gestiona `FileNotFoundError`, `EmptyDataError` y `ParserError`.

## Ejercicio 6
Construye un ejemplo donde `except:` oculte un error y refactorízalo.

## Ejercicio 7
Demuestra `try/except/else/finally`.

## Ejercicio 8
Crea `DatasetSchemaError`.

## Ejercicio 9
Relanza un error mediante `raise ... from`.

## Ejercicio 10
Construye un pipeline `validar → limpiar → transformar → analizar → exportar`.

## Ejercicio 11
Cuenta mensajes por nivel y grafícalos.

## Ejercicio 12 — Reto final
Refactoriza un script de al menos 40 líneas con:

- logger;
- FileHandler;
- niveles;
- excepción propia;
- captura específica;
- duración;
- error provocado;
- corrección;
- segunda ejecución exitosa.

# 27. Preguntas de reflexión

1. ¿Por qué `print()` no sustituye a logging?
2. ¿Cuándo usar DEBUG frente a INFO?
3. ¿Un WARNING debe detener el pipeline?
4. ¿Por qué evitar `except:`?
5. ¿Cuándo conviene relanzar?
6. ¿Qué aporta una excepción propia?
7. ¿Qué diferencia hay entre registrar y resolver un error?
8. ¿Para qué sirve `finally`?
9. ¿Qué ventaja ofrece `with`?
10. ¿Qué debería permitir reconstruir un buen log?